# Marvin — LLM path on Colab

This rig exercises **only** the language half of MVP.md: the system prompt, the
rolling context window, `<sigh>` interception, sentence streaming, and the
30-prompt cadence eval. No whisper, no Piper, no SoX, no V821.

**What it can tell you**

- Whether the nine cadence rules actually produce Marvin out of Qwen2.5-1.5B
- Whether sentence streaming emits clean sentence boundaries mid-generation
- Whether the context window keeps him in character across turns
- Phase 2's gate: >= 24/30 on the cadence eval

**What it cannot tell you**

Anything about latency. Colab is x86 with a different memory system; the
timings printed below are instrumentation, not measurements. Phase 3's
<= 4.0 s gate is only meaningful on the Pi 5.

## 1. Get the code

In [ ]:
# Either clone your remote, or upload the repo folder to /content/marvin.
REPO_URL = ""   # e.g. "https://github.com/you/marvin.git"

import os, pathlib
%cd /content
if REPO_URL and not pathlib.Path("/content/marvin").exists():
    !git clone --depth 1 $REPO_URL marvin
assert pathlib.Path("/content/marvin/pi/marvind").is_dir(), \
    "Upload the repo to /content/marvin, or set REPO_URL above."
%cd /content/marvin

## 2. Build llama-server and fetch the model

CPU-only build, on purpose. A CUDA build would make Colab behave nothing like a
Pi 5 and would hide the design pressure that sentence streaming exists to
relieve. First run takes roughly 5 minutes; re-runs skip straight through.

In [ ]:
!bash scripts/colab_setup.sh

## 3. Run the test suite

Pure functions, no model server needed. Run this before anything else — if the
splitter or the rule checks are broken, every number further down is noise.

In [ ]:
!python -m pytest tests/ --cov=marvind --cov=rules --cov=cadence_eval --cov-report=term-missing

## 4. Start llama-server

In [ ]:
!bash scripts/run_server.sh

## 5. One reply, streamed

Watch the sentences arrive one at a time. Each one is what would be handed to
Piper the moment it completes, rather than at the end of the completion. That
is mechanism 2 of MVP.md section 5, and it is why perceived latency and actual
latency come apart.

In [ ]:
import sys, time
sys.path.insert(0, "/content/marvin/pi")

from marvind import persona
from marvind.brain import stream_sentences, wait_for_server
from marvind.config import BrainConfig

config = BrainConfig(base_url="http://127.0.0.1:8080")
wait_for_server(config)
marvin = persona.load_persona(config.system_prompt_path, config.examples_path)

def ask(text, conversation=None):
    conversation = conversation or persona.Conversation(max_turns=config.max_turns)
    messages = persona.build_messages(marvin, conversation, text)
    started, first = time.perf_counter(), None
    parts = []
    for sentence in stream_sentences(config, messages):
        if first is None:
            first = time.perf_counter() - started
        print(f"  [{time.perf_counter() - started:5.2f}s] "
              f"{persona.render_segments(persona.intercept_sighs(sentence))}")
        parts.append(sentence)
    print(f"  first sentence {first:.2f}s, total {time.perf_counter() - started:.2f}s")
    return " ".join(parts)

_ = ask("Marvin, what's the weather like?")

## 6. Multi-turn, to test the rolling window

The cadence eval runs every prompt in a fresh context. This is where you find
out whether he holds character once there is history to drift against.

In [ ]:
conversation = persona.Conversation(max_turns=config.max_turns)
for line in ["Good morning, Marvin.", "Did you sleep well?", "What's two plus two?"]:
    print(f"you   > {line}")
    reply = ask(line, conversation)
    conversation = conversation.with_exchange(line, reply)
    print()

## 7. The cadence eval — Phase 2's gate

30 fixed prompts, scored per rule. Six of the nine rules are checked
mechanically; rules 1, 3 and 6 need an ear, so the printed score is a **screen**
until you fill in the audition sheet. Pin `--seed` so two runs are comparable.

In [ ]:
!python tools/eval/cadence_eval.py \
    --seed 42 \
    --out runs/latest.json \
    --audition runs/latest.md

## 8. Score the three rules a machine cannot

Read the sheet below aloud. Then edit `runs/latest.human.json` — `true`,
`false`, or `null` where a rule does not apply — and re-score. No model call is
made, so this is instant and can be repeated.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("runs/latest.md").read()))

In [ ]:
!python tools/eval/cadence_eval.py --score runs/latest.json --human runs/latest.human.json

## 9. Tuning loop

Everything that shapes the personality lives in three places:

- `pi/prompts/marvin.examples.json` — the few-shot turns, replayed to the model
  as real user/assistant messages. At 1.5B these do more work than the rules do,
  because the model imitates the assistant role far more reliably than it
  follows a description of it. Start here.
- `pi/prompts/marvin.system.md` — the nine rules and the formatting constraints.
- `BrainConfig` in `pi/marvind/config.py` — temperature, top_p, repeat_penalty,
  and `suppressed_tokens`, which bans a token at decode rather than asking the
  model not to use it.

Change one, re-run section 7 with the same `--seed`, and compare the per-rule
failure counts. Attributing a regression to a rule is the whole point of
reporting per rule rather than as a single number.

If rule 1 (lead with the complaint) stays above roughly ten failures once the
few-shots are in, the 1.5B is at its limit on register. That is the evidence
MVP.md section 10 wants before spending the RAM headroom on 3B — measured, not
guessed.